# ua-legal-lm — украинская legal-LM на GPU (Kaggle)

Обучение с нуля на открытых данных проекта [JoTalbot/ukraine](https://github.com/JoTalbot/ukraine):
ЄДРСР (метаданные судебных решений) + ЄДР (компании, учредители, статуты) + реестр ПДВ.

Ноутбук сам скачивает публичные Parquet-части с Hugging Face, строит корпус, обучает GPT
и (если добавлен секрет `HF_TOKEN`) публикует модель в HF Hub.


In [ ]:
# Конфигурация
YEARS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]   # какие годы ЄДРСР брать
PARTS_PER_YEAR = 8        # Parquet-частей (по 250k записей) на год
EDRSR_LIMIT = 3_000_000   # строк решений в корпус
EDR_LIMIT = 2_000_000     # компаний ЄДР
VAT_LIMIT = 400_000       # плательщиков ПДВ

DIM, LAYERS, CTX, BATCH, STEPS = 512, 8, 384, 16, 6000
VOCAB = 8192

HF_EDRSR = 'JoTalbot/ua-edrsr'
HF_OPEN = 'JoTalbot/ua-open-data'
HF_MODEL_REPO = 'JoTalbot/ua-legal-lm'   # куда публиковать модель
REPO = 'https://github.com/JoTalbot/ukraine'
print('config ok')


In [ ]:
# Установка и клонирование
# Kaggle может выдать Tesla P100 (sm_60); свежий torch поддерживает только sm_70+.
# Для P100 ставим torch 2.7.1+cu118 (последняя ветка с поддержкой Pascal).
# Обучение запускается subprocess'ом, поэтому подхватит новый torch без рестарта kernel.
!git clone -q {REPO} || true
!pip -q install tokenizers pyarrow huggingface_hub
import os, glob, json, subprocess, urllib.request
os.makedirs('data/edrsr', exist_ok=True)
import torch
print('preinstalled torch', torch.__version__, '| cuda avail:', torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    print('GPU:', name, '| capability:', cap)
    if cap[0] < 7:
        print('Pascal GPU: installing torch 2.7.1+cu118...')
        subprocess.run(['pip', 'install', '-q', 'torch==2.7.1',
                        '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
        r = subprocess.run(['python', '-c',
            'import torch;print("subprocess torch:", torch.__version__, torch.cuda.is_available())'],
            capture_output=True, text=True)
        print(r.stdout, r.stderr)


In [ ]:
# Скачивание данных с Hugging Face (публичные датасеты)
def ls(repo, prefix):
    url = f'https://huggingface.co/api/datasets/{repo}/tree/main/{prefix}'
    with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'kaggle'})) as r:
        return [i['path'] for i in json.load(r) if 'part-' in i['path']]

paths = []
for year in YEARS:
    parts = sorted(ls(HF_EDRSR, str(year)))[:PARTS_PER_YEAR]
    paths += parts
print('parts to download:', len(paths))
for i, p in enumerate(paths):
    dest = 'data/edrsr/' + p.replace('/', '_')
    if not os.path.exists(dest):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{HF_EDRSR}/resolve/main/{p}', dest)
    if (i + 1) % 10 == 0: print(f'  {i+1}/{len(paths)}')

urllib.request.urlretrieve(f'https://huggingface.co/datasets/{HF_OPEN}/resolve/main/edr/UO.zip', 'data/UO.zip')
urllib.request.urlretrieve(f'https://huggingface.co/datasets/{HF_OPEN}/resolve/main/vat_payers/pdv_actual.csv', 'data/pdv.csv')
print('downloaded:', len(glob.glob('data/edrsr/*')), 'parts + UO.zip + pdv.csv')


In [ ]:
# Сборка корпуса (стриминг, дедуп через sort -u)
!python ukraine/scripts/build_lm_corpus.py \
    --edrsr-parquet 'data/edrsr/*' \
    --edr-uo data/UO.zip \
    --vat data/pdv.csv \
    --edrsr-limit {EDRSR_LIMIT} --edr-limit {EDR_LIMIT} --vat-limit {VAT_LIMIT} \
    --output data/corpus.txt


In [ ]:
# Обучение GPT на GPU (traceback сохраняется в model/error.txt для автосбора)
import subprocess, pathlib, sys
cmd = [sys.executable, 'ukraine/scripts/train_lm.py',
       '--corpus', 'data/corpus.txt', '--out', 'model',
       '--device', 'cuda', '--vocab', str(VOCAB),
       '--dim', str(DIM), '--layers', str(LAYERS), '--ctx', str(CTX), '--batch', str(BATCH),
       '--steps', str(STEPS), '--eval-every', '250', '--sample-every', '500']
print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-4000:])
print(result.stderr[-4000:])
pathlib.Path('model/error.txt').write_text(
    'ok' if result.returncode == 0 else (result.stderr[-20000:] or 'no stderr'), encoding='utf-8')
print('training exit code:', result.returncode)


In [ ]:
# Очистка тяжёлых промежуточных файлов (output кернела = /kaggle/working)
import os, shutil
for p in ['data', 'model/tokens.uint16', 'model/corpus.sample.txt']:
    if os.path.exists(p):
        (shutil.rmtree if os.path.isdir(p) else os.remove)(p)
print('cleaned; remaining:', sorted(os.listdir('.')), sorted(os.listdir('model')) if os.path.isdir('model') else [])


In [ ]:
# Кривая обучения и примеры
import json
steps = [json.loads(l) for l in open('model/metrics.jsonl')]
val = [(s['step'], s['val_loss']) for s in steps if 'val_loss' in s]
print('val loss:', val[:3], '...', val[-3:])
try:
    import matplotlib.pyplot as plt
    tr = [(s['step'], s['loss']) for s in steps if 'loss' in s]
    plt.figure(figsize=(9, 4))
    plt.plot(*zip(*tr), label='train')
    plt.plot(*zip(*val), label='val')
    plt.legend(); plt.xlabel('step'); plt.ylabel('loss'); plt.show()
except Exception as e:
    print('no plots:', e)
print(open('model/samples.txt').read()[-2000:])


In [ ]:
# Публикация в Hugging Face Hub (нужен секрет HF_TOKEN в Kaggle)
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')

if token:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(repo_id=HF_MODEL_REPO, repo_type='model', exist_ok=True)
    for f in ['model/model.pt', 'model/tokenizer.json', 'model/metrics.jsonl', 'model/samples.txt']:
        api.upload_file(path_or_fileobj=f, path_in_repo=f.split('/')[-1],
                        repo_id=HF_MODEL_REPO, repo_type='model')
    print('published to', HF_MODEL_REPO)
else:
    print('HF_TOKEN не найден — модель сохранена локально в ./model')


## Замечания
- T4 (16 ГБ) хватает для dim 512 / layers 8 / ctx 384 / batch 16 (~29M параметров).
- 6000 шагов ≈ 1.5–2.5 ч; квота Kaggle — 30 GPU-часов/неделю.
- Увеличить корпус: `PARTS_PER_YEAR`, `EDRSR_LIMIT` (все 21 год лежат в `JoTalbot/ua-edrsr`).
- Приватность: только законно опубликованные открытые данные; обезличивание источника сохраняется.
